<a href="https://colab.research.google.com/github/kirti73/AI_lab/blob/main/1BF24CS140_lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### First-Order Logic (FOL) Resolution Prover Implementation

In [ ]:
import re

class Term:
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args is not None else []

    def is_variable(self):
        return len(self.args) == 0 and self.name[0].islower()

    def __eq__(self, other):
        if not isinstance(other, Term):
            return False
        return self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, tuple(self.args)))

    def __str__(self):
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(x) for x in self.args)})"

    def __repr__(self):
        return self.__str__()

class Literal:
    def __init__(self, predicate, args, is_negated=False):
        self.predicate = predicate
        self.args = [Term(arg) if isinstance(arg, str) else arg for arg in args]
        self.is_negated = is_negated

    def negate(self):
        return Literal(self.predicate, self.args, not self.is_negated)

    def __eq__(self, other):
        if not isinstance(other, Literal):
            return False
        return (self.predicate == other.predicate and
                self.args == other.args and
                self.is_negated == other.is_negated)

    def __hash__(self):
        return hash((self.predicate, tuple(self.args), self.is_negated))

    def __str__(self):
        sign = "~" if self.is_negated else ""
        args_str = f"({', '.join(str(x) for x in self.args)})" if self.args else ""
        return f"{sign}{self.predicate}{args_str}"

    def __repr__(self):
        return self.__str__()

class Clause:
    def __init__(self, literals):
        self.literals = frozenset(literals)

    def __eq__(self, other):
        if not isinstance(other, Clause):
            return False
        return self.literals == other.literals

    def __hash__(self):
        return hash(self.literals)

    def __str__(self):
        if not self.literals:
            return "[]"  # Contradiction / Empty clause
        return " | ".join(str(lit) for lit in sorted(self.literals, key=lambda l: str(l)))

    def __repr__(self):
        return self.__str__()

In [ ]:
def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    elif isinstance(val, Term) and val.is_variable() and val.name in subst:
        return unify(var, subst[val.name], subst)
    elif occur_check(var, val, subst):
        return None
    else:
        new_subst = subst.copy()
        new_subst[var] = val
        return new_subst

def occur_check(var, val, subst):
    if var == val.name:
        return True
    elif not val.is_variable() and val.args:
        for arg in val.args:
            if occur_check(var, arg, subst):
                return True
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is None:
        return None
    elif x == y:
        return subst
    elif isinstance(x, Term) and x.is_variable():
        return unify_var(x.name, y, subst)
    elif isinstance(y, Term) and y.is_variable():
        return unify_var(y.name, x, subst)
    elif isinstance(x, Term) and isinstance(y, Term) and not x.is_variable() and not y.is_variable():
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
        return subst
    elif isinstance(x, Literal) and isinstance(y, Literal):
        if x.predicate != y.predicate or len(x.args) != len(y.args) or x.is_negated == y.is_negated:
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
        return subst
    else:
        return None

def apply_subst_term(term, subst):
    if term.is_variable():
        if term.name in subst:
            val = subst[term.name]
            if isinstance(val, Term):
                return apply_subst_term(val, subst)
            return val
        return term
    else:
        return Term(term.name, [apply_subst_term(arg, subst) for arg in term.args])

def apply_subst_literal(lit, subst):
    return Literal(lit.predicate, [apply_subst_term(arg, subst) for arg in lit.args], lit.is_negated)

def apply_subst_clause(clause, subst):
    return Clause([apply_subst_literal(lit, subst) for lit in clause.literals])

In [ ]:
def resolve(clause1, clause2):
    resolvents = []
    for lit1 in clause1.literals:
        for lit2 in clause2.literals:
            if lit1.predicate == lit2.predicate and lit1.is_negated != lit2.is_negated:
                subst = unify(lit1, lit2)
                if subst is not None:
                    new_lits = (set(clause1.literals) - {lit1}) | (set(clause2.literals) - {lit2})
                    new_clause = Clause([apply_subst_literal(lit, subst) for lit in new_lits])
                    resolvents.append((new_clause, subst, lit1, lit2))
    return resolvents

def prove_by_resolution(kb_clauses, query_literal, max_steps=1000):
    negated_query = Clause([query_literal.negate()])
    clauses = list(kb_clauses) + [negated_query]

    print("--- Initial Clauses (including negated query) ---")
    for idx, c in enumerate(clauses):
        print(f"C{idx+1}: {c}")
    print("-------------------------------------------------\n")

    step = 0
    while step < max_steps:
        new_clauses_generated = False
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve(clauses[i], clauses[j])
                for res, subst, l1, l2 in resolvents:
                    if res not in clauses:
                        print(f"Resolving: {clauses[i]}  AND  {clauses[j]}")
                        print(f"  Unification: {subst}")
                        print(f"  Resolved Clause -> {res}\n")
                        if not res.literals:
                            print("Successfully found an empty clause [] (Contradiction)!")
                            print("Resolution Proof Succeeded: The query is TRUE.")
                            return True
                        clauses.append(res)
                        new_clauses_generated = True
        if not new_clauses_generated:
            break
        step += 1

    print("Resolution Proof Failed: Could not resolve to empty clause.")
    return False

In [8]:
import sys
import re

# --- 1. Base Logic Classes ---
class Term:
    def __init__(self, name, args=None):
        self.name = name
        self.args = args if args is not None else []

    def is_variable(self):
        return len(self.args) == 0 and self.name[0].islower()

    def __eq__(self, other):
        if not isinstance(other, Term):
            return False
        return self.name == other.name and self.args == other.args

    def __hash__(self):
        return hash((self.name, tuple(self.args)))

    def __str__(self):
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(x) for x in self.args)})"

    def __repr__(self):
        return self.__str__()

class Literal:
    def __init__(self, predicate, args, is_negated=False):
        self.predicate = predicate
        self.args = [Term(arg) if isinstance(arg, str) else arg for arg in args]
        self.is_negated = is_negated

    def negate(self):
        return Literal(self.predicate, self.args, not self.is_negated)

    def __eq__(self, other):
        if not isinstance(other, Literal):
            return False
        return (self.predicate == other.predicate and
                self.args == other.args and
                self.is_negated == other.is_negated)

    def __hash__(self):
        return hash((self.predicate, tuple(self.args), self.is_negated))

    def __str__(self):
        sign = "~" if self.is_negated else ""
        args_str = f"({', '.join(str(x) for x in self.args)})" if self.args else ""
        return f"{sign}{self.predicate}{args_str}"

    def __repr__(self):
        return self.__str__()

class Clause:
    _id_counter = 1
    def __init__(self, literals, parents=None, subst=None):
        self.literals = frozenset(literals)
        self.id = Clause._id_counter
        Clause._id_counter += 1
        self.parents = parents  # Tuple of (Clause1, Clause2)
        self.subst = subst      # Unification substitution dict

    def __eq__(self, other):
        if not isinstance(other, Clause):
            return False
        return self.literals == other.literals

    def __hash__(self):
        return hash(self.literals)

    def __str__(self):
        if not self.literals:
            return f"C{self.id}: []"
        return f"C{self.id}: " + " | ".join(str(lit) for lit in sorted(self.literals, key=lambda l: str(l)))

    def __repr__(self):
        return self.__str__()

# --- 2. Unification and Substitutions ---
def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    elif isinstance(val, Term) and val.is_variable() and val.name in subst:
        return unify(var, subst[val.name], subst)
    elif occur_check(var, val, subst):
        return None
    else:
        new_subst = subst.copy()
        new_subst[var] = val
        return new_subst

def occur_check(var, val, subst):
    if var == val.name:
        return True
    elif not val.is_variable() and val.args:
        for arg in val.args:
            if occur_check(var, arg, subst):
                return True
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is None:
        return None
    elif x == y:
        return subst
    elif isinstance(x, Term) and x.is_variable():
        return unify_var(x.name, y, subst)
    elif isinstance(y, Term) and y.is_variable():
        return unify_var(y.name, x, subst)
    elif isinstance(x, Term) and isinstance(y, Term) and not x.is_variable() and not y.is_variable():
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
        return subst
    elif isinstance(x, Literal) and isinstance(y, Literal):
        if x.predicate != y.predicate or len(x.args) != len(y.args) or x.is_negated == y.is_negated:
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
        return subst
    else:
        return None

def apply_subst_term(term, subst):
    if term.is_variable():
        if term.name in subst:
            val = subst[term.name]
            if isinstance(val, Term):
                return apply_subst_term(val, subst)
            return val
        return term
    else:
        return Term(term.name, [apply_subst_term(arg, subst) for arg in term.args])

def apply_subst_literal(lit, subst):
    return Literal(lit.predicate, [apply_subst_term(arg, subst) for arg in lit.args], lit.is_negated)

def apply_subst_clause(clause, subst):
    return Clause([apply_subst_literal(lit, subst) for lit in clause.literals], parents=clause.parents, subst=clause.subst)

# --- 3. Resolution, Tree Extraction and Proving ---
def resolve(clause1, clause2):
    resolvents = []
    for lit1 in clause1.literals:
        for lit2 in clause2.literals:
            if lit1.predicate == lit2.predicate and lit1.is_negated != lit2.is_negated:
                subst = unify(lit1, lit2)
                if subst is not None:
                    new_lits = (set(clause1.literals) - {lit1}) | (set(clause2.literals) - {lit2})
                    new_clause = Clause([apply_subst_literal(lit, subst) for lit in new_lits], parents=(clause1, clause2), subst=subst)
                    resolvents.append((new_clause, subst, lit1, lit2))
    return resolvents

def print_resolution_tree(clause, depth=0, visited=None):
    if visited is None:
        visited = set()
    if clause.id in visited:
        return
    visited.add(clause.id)

    indent = "  " * depth
    if clause.parents:
        p1, p2 = clause.parents
        subst_str = f" [Subst: {clause.subst}]" if clause.subst else ""
        print(f"{indent}├── {clause}{subst_str}")
        print_resolution_tree(p1, depth + 1, visited)
        print_resolution_tree(p2, depth + 1, visited)
    else:
        print(f"{indent}└── [Premise] {clause}")

def prove_by_resolution(kb_clauses, query_literal, max_steps=1000):
    # Reset counter for clean ID numbering
    Clause._id_counter = 1

    # Re-instantiate KB clauses to get clean sequential IDs
    fresh_kb = []
    for c in kb_clauses:
        fresh_kb.append(Clause(c.literals))

    negated_query = Clause([query_literal.negate()])
    clauses = list(fresh_kb) + [negated_query]

    print("--- Initial Clauses (including negated query) ---")
    for idx, c in enumerate(clauses):
        print(c)
    print("-------------------------------------------------\n")

    step = 0
    while step < max_steps:
        new_clauses_generated = False
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = resolve(clauses[i], clauses[j])
                for res, subst, l1, l2 in resolvents:
                    if res not in clauses:
                        print(f"Resolving: {clauses[i]}  AND  {clauses[j]}")
                        print(f"  Unification: {subst}")
                        print(f"  Resolved Clause -> {res}\n")
                        if not res.literals:
                            print("\n=========================================")
                            print("Successfully found an empty clause [] (Contradiction)!")
                            print("Resolution Proof Succeeded: The query is TRUE.\n")
                            print("--- Resolution Proof Tree (Traceback from []) ---")
                            print_resolution_tree(res)
                            print("=========================================")
                            return True
                        clauses.append(res)
                        new_clauses_generated = True
        if not new_clauses_generated:
            break
        step += 1

    print("Resolution Proof Failed: Could not resolve to empty clause.")
    return False

# --- 4. Robust Interactive Input Parser Helpers ---
def parse_literal_string(lit_str):
    lit_str = lit_str.strip()
    is_negated = False
    if lit_str.startswith('~'):
        is_negated = True
        lit_str = lit_str[1:]

    match = re.match(r"(\w+)\((.*)\)", lit_str)
    if not match:
        return Literal(lit_str, [], is_negated)

    predicate = match.group(1)
    args_str = match.group(2)
    args = [arg.strip() for arg in args_str.split(',') if arg.strip()]
    return Literal(predicate, args, is_negated)

def parse_clause_string(clause_str):
    lit_strs = clause_str.split('|')
    literals = [parse_literal_string(l) for l in lit_strs]
    return Clause(literals)

# Splits input strictly by newlines to keep formula spaces completely safe
def split_clauses(input_str):
    return [p.strip() for p in input_str.strip().split('\n') if p.strip()]

# --- 5. Interactive Execution Loop ---
print("=== FOL Resolution Prover Input Portal ===")
print("Instructions: Enter each CNF Clause line by line. Press enter twice or type 'DONE' to run.")
print("Separate literals within a clause with '|' (e.g., ~Food(x) | Likes(John, x)).")
print("Type 'DONE' or just press Enter without typing anything to finish and run.\n")

custom_kb = []

# Default case study inputs provided for fast copy-pasting
example_inputs = [
    "~Food(x) | Likes(John, x)",
    "Food(Apple)",
    "Food(Vegetables)",
    "~Eats(y, x) | Killed(y) | Food(x)",
    "Eats(Anil, Peanuts)",
    "Alive(Anil)",
    "~Eats(Anil, z) | Eats(Harry, z)",
    "~Alive(w) | ~Killed(w)"
]

print("--- Copy-Paste Examples for the Case Study premises ---")
for ex in example_inputs:
    print(ex)
print("------------------------------------------------------\n")

while True:
    user_input = input("Enter Clause(s): ").strip()
    if not user_input or user_input.upper() == 'DONE':
        break

    clauses_to_add = split_clauses(user_input)
    for cls_str in clauses_to_add:
        try:
            parsed_c = parse_clause_string(cls_str)
            custom_kb.append(parsed_c)
            print(f"-> Successfully Added Clause: {parsed_c}")
        except Exception as e:
            print(f"-> Error parsing clause '{cls_str}': {e}. Please try again.")
    print()

if not custom_kb:
    print("\nNo input detected! Defaulting to the pre-loaded Case Study knowledge base.")
    custom_kb = [
        parse_clause_string("~Food(x) | Likes(John, x)"),
        parse_clause_string("Food(Apple)"),
        parse_clause_string("Food(Vegetables)"),
        parse_clause_string("~Eats(y, x) | Killed(y) | Food(x)"),
        parse_clause_string("Eats(Anil, Peanuts)"),
        parse_clause_string("Alive(Anil)"),
        parse_clause_string("~Eats(Anil, z) | Eats(Harry, z)"),
        parse_clause_string("~Alive(w) | ~Killed(w)")
    ]

query_str = input("\nEnter the Query to Prove (e.g., 'Likes(John, Peanuts)'): ").strip()
if not query_str:
    query_str = "Likes(John, Peanuts)"

try:
    parsed_query = parse_literal_string(query_str)
    print(f"\nAnalyzing Proof for Query: {parsed_query}")
    print("==========================================")
    prove_by_resolution(custom_kb, parsed_query)
except Exception as e:
    print(f"Error parsing query: {e}")

=== FOL Resolution Prover Input Portal ===
Instructions: Enter each CNF Clause line by line. Press enter twice or type 'DONE' to run.
Separate literals within a clause with '|' (e.g., ~Food(x) | Likes(John, x)).
Type 'DONE' or just press Enter without typing anything to finish and run.

--- Copy-Paste Examples for the Case Study premises ---
~Food(x) | Likes(John, x)
Food(Apple)
Food(Vegetables)
~Eats(y, x) | Killed(y) | Food(x)
Eats(Anil, Peanuts)
Alive(Anil)
~Eats(Anil, z) | Eats(Harry, z)
~Alive(w) | ~Killed(w)
------------------------------------------------------

Enter Clause(s): ~SuffersAllergies(x) | Sneezes(x)
-> Successfully Added Clause: C1: Sneezes(x) | ~SuffersAllergies(x)

Enter Clause(s): ~LivesWith(x, y) | ~Cat(y) | ~AllergicTo(x, y) | SuffersAllergies(x)
-> Successfully Added Clause: C2: SuffersAllergies(x) | ~AllergicTo(x, y) | ~Cat(y) | ~LivesWith(x, y)

Enter Clause(s): Cat(Tom)
-> Successfully Added Clause: C3: Cat(Tom)

Enter Clause(s): ~Cat(y) | AllergicTo(Mary, 

### Case Study Translation: "Mary Sneezes"

**CNF Premises for the Prover:**
1. `~SuffersAllergies(x) | Sneezes(x)`
2. `~LivesWith(x, y) | ~Cat(y) | ~AllergicTo(x, y) | SuffersAllergies(x)`
3. `Cat(Tom)`
4. `~Cat(y) | AllergicTo(Mary, y)`
5. `LivesWith(Mary, Tom)`

**Query:**
`Sneezes(Mary)`

### Running the Solver for "Mary Sneezes"

Simply run the code cell `e0df596f` above, paste the 5 CNF clauses when asked for inputs (press enter after each), type `DONE` to submit, and then enter `Sneezes(Mary)` as the query to see the automatic resolution traceback tree!